# Project 5 — choosing an HR text-to-SQL model

Learn the flow in Six.md: requirements → budget → shortlist → custom evaluation → decision.

HR asks in plain language. The model writes SQL, SQLite runs it, and the notebook displays a table. We score the **answer table**, not the SQL text. The first version needs no web app or MySQL server.

Before opening this notebook, run `create_database.py` then `create_goldens.py`. Select the shared root `.venv` kernel in VS Code. Restart the kernel when switching projects or editing helpers.

## 1. Load the prepared inputs

models/settings.py defines paths; models/candidates.py loads the four active candidates; models/client.py calls Gemini or Groq. tools/evaluation.py compares answer tables and tools/results.py saves reports. Preparation stays outside this notebook.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

folder = Path.cwd()
if folder.name != '5_hr_text_to_sql':
    folder = folder / '5_hr_text_to_sql'
assert folder.is_dir(), 'Open from the repository root or Project 5 folder.'
os.chdir(folder)
sys.path.insert(0, str(folder.resolve()))

from models.settings import PROJECT, DATA
from models.client import build_prompt, call_model
from models.candidates import load_selection
from tools.evaluation import load_goldens, execute_sql, same_answer, evaluate, summarize
from tools.results import save_run, load_latest_reports, save_comparison

cases = load_goldens()
pd.DataFrame([{k: case[k] for k in ['id', 'difficulty', 'question']} for case in cases])

,id,difficulty,question
0,q001,simple,"List all department IDs and department names, ..."
1,q002,simple,"List the employee ID, first name, last name, a..."
2,q003,intermediate,"As of 2026-10-01, show every department name, ..."
3,q004,intermediate,"As of 2026-10-01, which current Development em..."
4,q005,difficult,"As of 2026-10-01, list current employees at th..."
5,q006,difficult,"As of 2026-10-01, which current employees rece..."


## 2. Inspect one golden answer

There are two simple, two intermediate, and two difficult questions. Goldens contain reference SQL and the saved results. They were prepared by AI and checked against independently calculated values; human review is pending. **Only schema, business rules, and the question go to the model.**

In [2]:
case = cases[2]
print(case['question'])
print(case['reference_sql'])
pd.DataFrame(case['rows'], columns=case['columns'])

As of 2026-10-01, show every department name, its current employee count, and average current annual salary rounded to two decimals. Include empty departments with count 0 and average NULL. Order by department name.
SELECT d.dept_name, COUNT(de.emp_no) AS employee_count,
       ROUND(AVG(s.salary), 2) AS average_salary
FROM departments d
LEFT JOIN dept_emp de ON de.dept_no = d.dept_no
 AND de.from_date <= '2026-10-01' AND de.to_date > '2026-10-01'
LEFT JOIN salaries s ON s.emp_no = de.emp_no
 AND s.from_date <= '2026-10-01' AND s.to_date > '2026-10-01'
GROUP BY d.dept_no, d.dept_name ORDER BY d.dept_name;


,dept_name,employee_count,average_salary
0,Development,7,92142.86
1,Finance,5,97000.00
2,Human Resources,3,83333.33
3,Research,0,NaN
4,Sales,5,94000.00


## 3. Budget and models available through your keys

Use the existing Gemini and Groq keys only. The active candidates are GPT-OSS 120B, GPT-OSS 20B, Qwen 3.8 27B (preview), and Gemini 3.5 Flash-Lite. Both authenticated model listings included these IDs. This is a key-constrained comparison, separate from the initial all-provider leaderboard shortlist.

The same budget assumptions apply: 1,000 requests/day, 30 days, 1,600 input and 400 output tokens, INR 95/USD, and 20% reserve. The four providers/models below fit the planning budget. Actual tokens and latency will be measured in Step 6.

In [3]:
selection = load_selection()
print(selection["selection_basis"])
display(pd.Series(selection["budget"]))
pd.DataFrame(selection["shortlisted"])[["name", "provider", "reasoning_effort", "input_price", "output_price", "monthly_inr"]]

User requested existing Gemini and Groq keys only. Verified IDs with both authenticated model-list endpoints on 2026-10-15. Four varied candidates for the same six-question pilot; this is not the original all-provider leaderboard top four.


monthly_budget_inr                                                       10000
requests_per_day                                                          1000
days_per_month                                                              30
input_tokens_per_request                                                  1600
output_tokens_per_request                                                  400
inr_per_usd                                                                 95
contingency_fraction                                                       0.2
quality_weight                                                             0.9
speed_weight                                                               0.1
target_seconds                                                               3
notes                        Planning conversion, not live FX. No caching, ...
dtype: object

,name,provider,reasoning_effort,input_price,output_price,monthly_inr
0,GPT-OSS 120B (Groq),groq,low,0.150,0.6,1641.6
1,GPT-OSS 20B (Groq),groq,low,0.075,0.3,820.8
2,Qwen 3.8 27B (Groq),groq,none,0.800,4.0,9849.6
3,Gemini 3.5 Flash-Lite (Google),gemini,minimal,0.300,2.5,5061.6


## 4. Verify the evaluator without an API

Execute all six reference queries and expect 6/6. This checks the fixture and evaluator; it is not a model benchmark.

In [4]:
checks = [{'id': c['id'], 'correct': same_answer(execute_sql(c['reference_sql']), c)} for c in cases]
assert all(row['correct'] for row in checks)
pd.DataFrame(checks)

,id,correct
0,q001,True
1,q002,True
2,q003,True
3,q004,True
4,q005,True
5,q006,True


## 5. Ask an HR question

The existing root .env supplies GROQ_API_KEY and GOOGLE_API_KEY. RUN_LIVE controls this optional single-question demonstration. Step 6 has its own RUN_STEP6 switch so we can evaluate without making an extra demo call.

In [5]:
RUN_LIVE = False
model_id = selection["shortlisted"][0]["model_id"]
question = "List all department IDs and department names, ordered by department name."
if RUN_LIVE:
    response = call_model(model_id, question)
    print(response["sql"])
    answer = execute_sql(response["sql"])
    print(answer["error"])
    display(pd.DataFrame(answer["rows"], columns=answer["columns"]))
else:
    print("Optional demo skipped; Step 6 uses RUN_STEP6.")

Optional demo skipped; Step 6 uses RUN_STEP6.


## 6. Compare four models using your existing keys

The completed live comparison used 24 successful calls: six questions per model. Matching result tables score 1; wrong tables or SQL errors score 0. Saved reports include measured token usage, estimated cost, and end-to-end latency.

GPT-OSS uses low reasoning, Qwen uses none, and Gemini 3.5 Flash-Lite uses minimal thinking. All allow 2,048 output tokens. Three-second pauses between requests help avoid rate limits; pauses are excluded from single-request latency. There are no automatic retries or silent model fallbacks.

The default RUN_STEP6=False displays saved live reports without spending tokens. Set it to True to make a fresh 24-call comparison. Gemini 3.8 returned high-demand errors and 2.5 Flash returned 404; we explicitly selected the working 3.5 Flash-Lite instead.

In [6]:
RUN_STEP6 = False  # True makes 24 fresh API calls; False displays the saved live results.
if RUN_STEP6:
    for model in selection["shortlisted"]:
        records = evaluate(cases, lambda question: call_model(model["model_id"], question), request_pause_seconds=3)
        summary = summarize(records, model, selection["budget"])
        report = save_run(model["model_id"], records, summary, {"model": model, "budget": selection["budget"], "as_of": "2026-10-01", "selection_basis": selection["selection_basis"], "request_pause_seconds": 3})
        print(report)
reports = load_latest_reports(selection["shortlisted"])
summaries = save_comparison(reports)
comparison = pd.DataFrame(summaries)
display(comparison[["model", "questions", "accuracy", "sql_errors", "mean_seconds", "projected_monthly_inr", "estimated_run_usd"]])

,model,questions,accuracy,sql_errors,mean_seconds,projected_monthly_inr,estimated_run_usd
0,GPT-OSS 120B (Groq),6,1.000000,0,1.017733,816.696,0.001433
1,Gemini 3.5 Flash-Lite (Google),6,1.000000,0,3.088150,2706.531,0.004748
2,Qwen 3.8 27B (Groq),6,1.000000,0,0.724783,4758.816,0.008349
3,GPT-OSS 20B (Groq),6,0.833333,0,0.888267,418.095,0.000733


## 7. Inspect failures and make a decision

Inspect docs/evaluatedds for generated SQL, expected rows, actual rows, and SQL errors. A 5/6 score is 83.3%; one example changes accuracy by 16.7 percentage points.

Aim for 6/6, measured projected monthly cost within INR 10,000, and mean end-to-end latency near three seconds. Among candidates that pass, prefer lower cost, then lower latency. If none pass, report that outcome.

These six AI-prepared goldens still need human review and more holdout questions before a production decision. Change one setting and compare timestamped reports.

Observed in this saved run: GPT-OSS 120B, Qwen 3.8 27B, and Gemini 3.5 Flash-Lite matched 6/6 answers. GPT-OSS 20B matched 5/6: q006 omitted the >=10% raise filter, returning 19 rows instead of nine.

GPT-OSS 120B is the preferred pilot candidate: 6/6, about one second per request, and about INR 817/month at measured token usage. This is a six-question finding, not a production reliability guarantee. All goldens still need human review.